# Notebook 13 - SSL Embeddings Exploration

## Goal
Inspect frame embeddings from pre-trained SSL speech models.


## Agenda
- Load audio
- Run wav2vec2 with hidden states
- Compare layer activation stats
- Pool embeddings


## Concept and Math

SSL models provide high-dimensional contextualized features.
Different layers capture different granularity from acoustic to higher-level structure.


In [ ]:
from pathlib import Path
import librosa as lb
import matplotlib.pyplot as plt
import torch
from transformers import AutoFeatureExtractor, Wav2Vec2Model

DATA_ROOT = Path("../dataset")
audio_files = sorted(DATA_ROOT.rglob("*.flac")) + sorted(DATA_ROOT.rglob("*.wav"))
if not audio_files:
    raise FileNotFoundError("No audio found under ../dataset")

wave, _ = lb.load(audio_files[0], sr=16000, mono=True)
model_name = "facebook/wav2vec2-base"
extractor = AutoFeatureExtractor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name).eval()

inputs = extractor(wave, sampling_rate=16000, return_tensors="pt")
with torch.no_grad():
    out = model(**inputs, output_hidden_states=True)

means = [h.abs().mean().item() for h in out.hidden_states]
plt.plot(means, marker="o")
plt.title("Mean absolute activation per layer")
plt.xlabel("Layer")
plt.ylabel("Mean |activation|")
plt.show()


## PyTorch Equivalent Snippet
Understand the librosa block first, then map it to this snippet.


In [ ]:
# transformers is already torch-native.
last = out.hidden_states[-1]  # [batch, time, dim]
pooled = last.mean(dim=1)
print(pooled.shape)


## Review Checklist
- Why inspect multiple layers instead of only the last layer?
- How can pooled embeddings be used in classifiers?
- What does layer activation magnitude indicate?
